# 🚀 Ray Local Tasks/Actors

<div align="center">
  <img src="./rayBasicPatterns.png" width="250">
</div>

### **Template Review**
This template introduces the core primitives of **Ray**: **Tasks** (Stateless functions) and **Actors** (Stateful classes). Optimized for **Saturn Cloud Jupyter Notebooks**, it demonstrates how to parallelize standard Python code across all available CPU cores on a single instance.

**Core Workflow:** We will compare serial Python execution with **Ray Tasks** for parallel work and use **Ray Actors** to manage a shared distributed state.

### **Tech Stack**
* **Ray**: Distributed computing framework.
* **Infrastructure**: [Saturn Cloud](https://saturncloud.io/) CPU Jupyter Instance.

In [ ]:
# Install Ray
!pip install ray -q

### **Step 1: Initialize Ray Locally**
Running `ray.init()` starts a local scheduler and worker processes on your machine. It will automatically detect the number of CPUs available.

In [ ]:
import ray
import time

# Initialize Ray on the local machine
if ray.is_initialized():
    ray.shutdown()
ray.init()

### **Step 2: Ray Tasks (Stateless Parallelism)**
Tasks are functions decorated with `@ray.remote`. They are executed asynchronously and return 'Object Refs' (futures).

In [ ]:
# A standard Python function
def slow_function(i):
    time.sleep(1)
    return i * i

# A Ray Task
@ray.remote
def remote_slow_function(i):
    time.sleep(1)
    return i * i

print("🐢 Running Serially...")
start = time.time()
results_serial = [slow_function(i) for i in range(4)]
print(f"Serial Duration: {time.time() - start:.2f}s")

print("\n🐇 Running Parallel Ray Tasks...")
start = time.time()
# Task calls return futures immediately
futures = [remote_slow_function.remote(i) for i in range(4)]
# ray.get() blocks until all tasks are finished
results_ray = ray.get(futures)
print(f"Ray Task Duration: {time.time() - start:.2f}s")
print(f"Results: {results_ray}")

### **Step 3: Ray Actors (Stateful Parallelism)**
Actors are classes decorated with `@ray.remote`. They allow you to maintain state (like a counter or a model) across multiple tasks in a distributed environment.

In [ ]:
@ray.remote
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1
        return self.value

    def get_count(self):
        return self.value

# Instantiate the actor
counter = Counter.remote()

print("🚀 Incrementing actor state from multiple calls...")
results = ray.get([counter.increment.remote() for _ in range(5)])
final_count = ray.get(counter.get_count.remote())

print(f"Progressive increments: {results}")
print(f"Final Actor State: {final_count}")

---
## Conclusion & Next Steps
We have successfully used **Ray Tasks** to parallelize functions and **Ray Actors** to manage state across a distributed system. Ray is designed to scale seamlessly from local CPU to a cluster of thousands of nodes.

### **Resources**
* **Cloud Infrastructure**: [Deploy on Saturn Cloud](https://saturncloud.io/)
* **Ray Documentation**: [Key Concepts](https://docs.ray.io/en/latest/ray-core/concepts.html)